# LeetCode 1034 — Coloring A Border bằng Python

Link đề: https://leetcode.com/problems/coloring-a-border/ · Độ khó: **Medium** · Chủ đề: Array (mảng), DFS, BFS, Matrix (ma trận)

Skeleton LeetCode:

```python
class Solution:
    def colorBorder(self, grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
        ...
```

Ghi chú: cho ma trận màu `grid`, một ô xuất phát `(row, col)` và một màu `color`. **Thành phần liên thông** của ô xuất phát gồm các ô **cùng màu** nối với nhau qua **4 hướng**. **Biên** của thành phần là các ô của nó **nằm ở mép lưới** hoặc **kề (4 hướng) một ô không thuộc thành phần**. Hãy tô các ô biên đó bằng `color` và trả về lưới mới. Bài này luyện **loang** (flood fill) rồi **tô sau**: cạm bẫy lớn nhất là **tô màu ngay khi đang loang**, làm các ô lân cận nhìn thấy màu mới và tưởng ô đó **không thuộc** thành phần. Kết quả là một lưới **duy nhất**, nên so sánh đúng từng ô.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thành phần liên thông (cùng màu, 4 hướng), định nghĩa **biên** gồm hai điều kiện (mép lưới, hoặc kề ô ngoài thành phần).
2. Nắm các công cụ Python cần thiết: lưới dạng danh sách hai chiều, bốn hướng kề, `deque` cho BFS, ngăn xếp cho DFS lặp, `set` các ô thăm, sao chép sâu `[row[:] for row in grid]`.
3. Hiểu nguyên lý: **tách hai pha**: (1) tìm toàn bộ thành phần (dựa vào màu **gốc**); (2) xác định biên và **tô** sau cùng (hoặc ghi nhớ các ô biên rồi tô cuối).
4. Đi qua 4 cách: dựng tập thành phần rồi kiểm tra từng ô theo đúng định nghĩa (brute force), DFS lặp có tập `visited`, BFS có tập `visited` (cách nộp chính), và đánh dấu tạm ngay trong lưới rồi quét lại.
5. Tránh các lỗi kinh điển: tô màu **trong lúc loang**, quên rằng ô ở **mép lưới** luôn là biên, chỉ xét một vài hướng, loang **toàn bộ** ô cùng màu trên lưới (không chỉ thành phần liên thông), đi 8 hướng, và DFS **đệ quy** quá sâu (lưới `50 x 50` có thể sâu hơn `1000`).

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `m x n` tên `grid` và ba số nguyên `row`, `col`, `color`. Mỗi giá trị trong lưới là màu của ô tại vị trí đó.

Hai ô **kề nhau** nếu chúng cạnh nhau theo một trong **4 hướng**. Hai ô thuộc cùng một **thành phần liên thông** nếu chúng có **cùng màu** và kề nhau (trực tiếp hoặc qua chuỗi ô cùng màu).

**Biên của một thành phần liên thông** gồm mọi ô của thành phần thỏa một trong hai điều kiện: kề (ít nhất một) ô **không thuộc thành phần**, hoặc nằm ở **mép lưới** (hàng đầu/cuối hoặc cột đầu/cuối).

Hãy tô **biên** của thành phần liên thông chứa ô `grid[row][col]` bằng màu `color`. Trả về lưới cuối cùng.

**Ví dụ 1:** `grid = [[1,1],[1,2]]`, `row = 0`, `col = 0`, `color = 3` -> `[[3,3],[3,2]]`.

**Ví dụ 2:** `grid = [[1,2,2],[2,3,2]]`, `row = 0`, `col = 1`, `color = 3` -> `[[1,3,3],[2,3,3]]`.

**Ví dụ 3:** `grid = [[1,1,1],[1,1,1],[1,1,1]]`, `row = 1`, `col = 1`, `color = 2` -> `[[2,2,2],[2,1,2],[2,2,2]]`.

**Ràng buộc:**

- `m == grid.length`, `n == grid[i].length`
- `1 <= m, n <= 50`
- `1 <= grid[i][j], color <= 1000`
- `0 <= row < m`, `0 <= col < n`

In [1]:
from typing import List, Set, Tuple
from collections import deque
import copy
import random, time

DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

examples = [
    ([[1, 1], [1, 2]], 0, 0, 3, [[3, 3], [3, 2]]),
    ([[1, 2, 2], [2, 3, 2]], 0, 1, 3, [[1, 3, 3], [2, 3, 3]]),
    ([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 1, 1, 2, [[2, 2, 2], [2, 1, 2], [2, 2, 2]]),
]
for grid, r, c, color, expected in examples:
    print(f"grid={grid} (row,col)=({r},{c}) color={color} -> kỳ vọng {expected}")

grid=[[1, 1], [1, 2]] (row,col)=(0,0) color=3 -> kỳ vọng [[3, 3], [3, 2]]
grid=[[1, 2, 2], [2, 3, 2]] (row,col)=(0,1) color=3 -> kỳ vọng [[1, 3, 3], [2, 3, 3]]
grid=[[1, 1, 1], [1, 1, 1], [1, 1, 1]] (row,col)=(1,1) color=2 -> kỳ vọng [[2, 2, 2], [2, 1, 2], [2, 2, 2]]


## 3. Kiến thức nền

### 3.1. Lưới, ô kề và kiểm tra biên lưới

Ô `(i, j)` có bốn ô kề `(i + di, j + dj)`. Ô kề phải nằm trong lưới: `0 <= a < m` và `0 <= b < n`. Ô ở **mép lưới** khi `i in (0, m - 1)` hoặc `j in (0, n - 1)`: ô đó luôn thuộc biên của thành phần (theo đề), dù mọi ô kề trong lưới cùng màu.

In [2]:
grid = examples[2][0]
m, n = len(grid), len(grid[0])
def on_edge(i, j): return i in (0, m - 1) or j in (0, n - 1)
print([[int(on_edge(i, j)) for j in range(n)] for i in range(m)], "(1 = ở mép lưới)")

[[1, 1, 1], [1, 0, 1], [1, 1, 1]] (1 = ở mép lưới)


### 3.2. Loang (flood fill) và vì sao cần màu gốc

Loang từ `(row, col)` qua các ô kề có **cùng màu gốc** `orig = grid[row][col]`, dùng tập `visited` để không thăm lại. Thành phần là tập các ô đã thăm. Quan trọng: so sánh với `orig` (màu **trước khi tô**) và **chưa tô** gì trong lúc loang; nếu tô sớm, ô đã tô có màu mới `!= orig` và các ô lân cận sẽ tưởng nó **không** thuộc thành phần.

### 3.3. Hai pha: tìm thành phần, rồi xác định biên và tô

Pha 1 cho tập thành phần `comp`. Pha 2: với mỗi ô trong `comp`, ô là biên nếu ở mép lưới **hoặc** có ô kề (trong lưới) **không** thuộc `comp`. Tô tất cả ô biên **sau cùng** (hoặc ghi vào bản sao của lưới, còn việc kiểm tra dùng `comp`).

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: lưới `3 x 3` toàn `1`, xuất phát `(1, 1)`, `color = 2`. Thành phần là cả `9` ô. Mọi ô ở mép lưới (8 ô) là biên (mép lưới). Ô giữa `(1, 1)`: cả bốn ô kề đều thuộc thành phần và nó không ở mép ⇒ **không** là biên, giữ nguyên `1`. Kết quả `[[2,2,2],[2,1,2],[2,2,2]]`.

Ví dụ 2: `grid = [[1,2,2],[2,3,2]]`, xuất phát `(0, 1)` (màu `2`). Thành phần: `(0,1), (0,2), (1,2)` (các ô `2` nối nhau; ô `(1,0)` cũng là `2` nhưng không kề thành phần). Cả ba ô đều ở mép lưới ⇒ đều là biên: tô thành `3`. Ô `(1,0)` không thuộc thành phần nên giữ `2`. Kết quả `[[1,3,3],[2,3,3]]`.

## 5. Cách 1 — Brute Force: dựng thành phần rồi kiểm tra từng ô theo định nghĩa

**Ý tưởng.** Pha 1: BFS dựng tập thành phần `comp`. Pha 2: với **từng** ô của `comp`, kiểm tra **đúng theo định nghĩa**: ở mép lưới, hoặc tồn tại một trong **bốn** ô kề (trong lưới) không thuộc `comp`. Tô các ô biên trên **bản sao** của lưới. Làm đúng định nghĩa của đề nên là **oracle độc lập**.

In [3]:
def color_border_brute(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    comp = {(row, col)}
    dq = deque([(row, col)])
    while dq:
        i, j = dq.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == orig and (a, b) not in comp:
                comp.add((a, b))
                dq.append((a, b))
    res = [r[:] for r in grid]
    for i, j in comp:
        at_edge = i in (0, m - 1) or j in (0, n - 1)
        touches_outside = any(0 <= i + di < m and 0 <= j + dj < n and (i + di, j + dj) not in comp for di, dj in DIRS)
        if at_edge or touches_outside:
            res[i][j] = color
    return res

for grid, r, c, color, expected in examples:
    got = color_border_brute(grid, r, c, color)
    print(f"brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

brute = [[3, 3], [3, 2]] OK
brute = [[1, 3, 3], [2, 3, 3]] OK
brute = [[2, 2, 2], [2, 1, 2], [2, 2, 2]] OK


**Độ phức tạp và nút thắt.** `O(m * n)` thời gian (mỗi ô của thành phần xét 4 ô kề), `O(m * n)` bộ nhớ. Đây vốn đã tuyến tính nên "nút thắt" không phải tốc độ mà là **tính đúng**: khi viết gọn thành một lượt DFS vừa loang vừa tô, ta rất dễ rơi vào lỗi tô sớm. Ta viết lại thành các cách gọn hơn, giữ hai pha.

## 6. Ý tưởng thuật toán / bước đột phá

**Giữ nguyên lưới gốc trong lúc loang; ghi nhận ô biên; tô sau.**

- Loang từ `(row, col)` với tập `visited`; so sánh màu với `orig`.
- Trong lúc thăm một ô `(i, j)`, đếm các ô kề: nếu ô kề nằm ngoài lưới **hoặc** có màu khác `orig` thì `(i, j)` là biên (ghi vào danh sách `border`).
  - Chú ý: ô kề **cùng màu `orig`** nhưng có thể chưa thăm vẫn thuộc thành phần (kề nhau, cùng màu), nên không làm `(i, j)` thành biên.
  - Ô ở mép lưới có ô kề ngoài lưới nên được xác định là biên bởi điều kiện "ô kề ngoài lưới".
- Sau khi loang xong, tô mọi ô trong `border` bằng `color`.

| Cách | Ý tưởng |
|---|---|
| 2 | DFS lặp (ngăn xếp) + `visited`, ghi danh sách biên, tô sau |
| 3 | BFS + `visited`, ghi danh sách biên, tô sau (nộp) |
| 4 | Đánh dấu tạm các ô thành phần bằng giá trị âm trong lưới rồi quét lại: ô thuộc biên được tô `color`, ô trong được khôi phục |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DFS lặp với `visited`

**Giải thích từng đoạn.**

- Ngăn xếp `stack = [(row, col)]`, `visited = {(row, col)}`.
- Với mỗi ô lấy ra: duyệt bốn ô kề. Nếu ô kề **ngoài lưới** hoặc **khác màu `orig`**: ô hiện tại là biên (ghi một lần). Nếu ô kề cùng màu `orig` và chưa thăm: thêm vào ngăn xếp.
- Dùng ngăn xếp tường minh vì thành phần có thể dài tới `m * n = 2500` ô (vượt giới hạn đệ quy mặc định `1000`).
- Cuối cùng tô `color` cho các ô biên trên bản sao.

In [4]:
def color_border_dfs(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    stack = [(row, col)]
    border = []
    while stack:
        i, j = stack.pop()
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                is_border = True                          # ô kề ngoài lưới hoặc khác màu gốc
            elif (a, b) not in visited:
                visited.add((a, b))
                stack.append((a, b))
        if is_border:
            border.append((i, j))
    res = [r[:] for r in grid]
    for i, j in border:
        res[i][j] = color
    return res

for grid, r, c, color, expected in examples:
    got = color_border_dfs(grid, r, c, color)
    print(f"DFS lặp = {got}", "OK" if got == expected else "SAI")
    assert got == expected

DFS lặp = [[3, 3], [3, 2]] OK
DFS lặp = [[1, 3, 3], [2, 3, 3]] OK
DFS lặp = [[2, 2, 2], [2, 1, 2], [2, 2, 2]] OK


**Code nộp LeetCode (Cách 2):**

In [5]:
class Solution:
    def colorBorder(self, grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
        m, n = len(grid), len(grid[0])
        orig = grid[row][col]
        visited = {(row, col)}
        stack = [(row, col)]
        border = []
        while stack:
            i, j = stack.pop()
            is_border = False
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                    is_border = True                  # kề ô ngoài lưới hoặc ô khác màu
                elif (a, b) not in visited:
                    visited.add((a, b))
                    stack.append((a, b))
            if is_border:
                border.append((i, j))
        for i, j in border:                           # tô SAU khi loang xong
            grid[i][j] = color
        return grid

sol_dfs = Solution().colorBorder
for grid, r, c, color, expected in examples:
    assert sol_dfs(copy.deepcopy(grid), r, c, color) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(m * n)`, bộ nhớ `O(m * n)` (tập `visited`, ngăn xếp, danh sách biên).

---

### 7.2. Cách 3 — BFS với `visited` (cách nộp chính)

**Giải thích từng đoạn.** Giống Cách 2 nhưng dùng hàng đợi `deque`. Thứ tự duyệt khác nhưng tập thành phần và tập biên **như nhau** (không phụ thuộc thứ tự). Sao chép lưới trước khi tô để không sửa đầu vào (ở bản nộp ta sửa trực tiếp `grid` vì đề cho phép và tiết kiệm bộ nhớ).

In [6]:
def color_border_bfs(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    dq = deque([(row, col)])
    border = []
    while dq:
        i, j = dq.popleft()
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                is_border = True
            elif (a, b) not in visited:
                visited.add((a, b))
                dq.append((a, b))
        if is_border:
            border.append((i, j))
    res = [r[:] for r in grid]
    for i, j in border:
        res[i][j] = color
    return res

for grid, r, c, color, expected in examples:
    got = color_border_bfs(grid, r, c, color)
    print(f"BFS = {got}", "OK" if got == expected else "SAI")
    assert got == expected

BFS = [[3, 3], [3, 2]] OK
BFS = [[1, 3, 3], [2, 3, 3]] OK
BFS = [[2, 2, 2], [2, 1, 2], [2, 2, 2]] OK


**Code nộp LeetCode (Cách 3):**

In [7]:
class Solution:
    def colorBorder(self, grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
        from collections import deque
        m, n = len(grid), len(grid[0])
        orig = grid[row][col]
        visited = {(row, col)}
        dq = deque([(row, col)])
        border = []
        while dq:
            i, j = dq.popleft()
            is_border = False
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                    is_border = True                  # kề ô ngoài lưới hoặc ô khác màu: ô này là biên
                elif (a, b) not in visited:
                    visited.add((a, b))
                    dq.append((a, b))
            if is_border:
                border.append((i, j))
        for i, j in border:                           # chỉ tô khi đã loang xong
            grid[i][j] = color
        return grid

sol_bfs = Solution().colorBorder
for grid, r, c, color, expected in examples:
    assert sol_bfs(copy.deepcopy(grid), r, c, color) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(m * n)`, bộ nhớ `O(m * n)`.

---

### 7.3. Cách 4 — Đánh dấu tạm ngay trong lưới

**Ý tưởng khác.** Không dùng `visited` riêng: loang và **đánh dấu** các ô của thành phần bằng một giá trị **tạm** `-orig` (âm, không trùng màu nào vì mọi màu `>= 1`), cùng lúc tránh thăm lại. Sau khi loang xong, quét danh sách các ô thành phần: ô thuộc biên (mép lưới, hoặc có ô kề trong lưới **không** mang dấu tạm) được tô `color`; ô còn lại (nằm trong) được **khôi phục** về `orig`.

**Giải thích từng đoạn.**

- Dấu tạm `-orig` phân biệt ô thành phần với mọi ô khác (kể cả các ô cùng màu `orig` nhưng thuộc thành phần khác: chúng vẫn là `orig` dương).
- Khi kiểm tra biên: ô kề trong lưới có giá trị `!= -orig` ⇒ không thuộc thành phần ⇒ ô hiện tại là biên.
- Cuối cùng không còn giá trị âm trong lưới.

In [8]:
def color_border_marker(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    g = [r[:] for r in grid]
    m, n = len(g), len(g[0])
    orig = g[row][col]
    mark = -orig
    g[row][col] = mark
    comp = [(row, col)]
    k = 0
    while k < len(comp):                               # BFS dùng chính danh sách làm hàng đợi
        i, j = comp[k]
        k += 1
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and g[a][b] == orig:
                g[a][b] = mark
                comp.append((a, b))
    border = [(i, j) for i, j in comp                      # xác định biên khi lưới còn nguyên dấu tạm
              if i in (0, m - 1) or j in (0, n - 1)
              or any(g[i + di][j + dj] != mark for di, dj in DIRS)]
    for i, j in comp:                                      # khôi phục thành phần về màu gốc
        g[i][j] = orig
    for i, j in border:                                    # rồi mới tô biên
        g[i][j] = color
    return g

for grid, r, c, color, expected in examples:
    got = color_border_marker(grid, r, c, color)
    print(f"đánh dấu tạm = {got}", "OK" if got == expected else "SAI")
    assert got == expected

đánh dấu tạm = [[3, 3], [3, 2]] OK
đánh dấu tạm = [[1, 3, 3], [2, 3, 3]] OK
đánh dấu tạm = [[2, 2, 2], [2, 1, 2], [2, 2, 2]] OK


Lưu ý: ta **xác định hết các ô biên trước** (lưới còn nguyên dấu tạm), rồi mới khôi phục và tô. Nếu vừa kiểm tra vừa ghi lại giá trị thì một ô đã xử lý không còn mang dấu tạm, các ô xử lý sau sẽ nhìn nó như "ngoài thành phần" và xác định biên sai.

**Code nộp LeetCode (Cách 4):**

In [9]:
class Solution:
    def colorBorder(self, grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
        m, n = len(grid), len(grid[0])
        orig = grid[row][col]
        mark = -orig                                   # dấu tạm: mọi màu đều >= 1 nên mark không trùng màu nào
        grid[row][col] = mark
        comp = [(row, col)]
        k = 0
        while k < len(comp):                           # BFS: danh sách làm hàng đợi
            i, j = comp[k]
            k += 1
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if 0 <= a < m and 0 <= b < n and grid[a][b] == orig:
                    grid[a][b] = mark
                    comp.append((a, b))
        border = []
        for i, j in comp:                              # xác định biên khi lưới còn nguyên dấu tạm
            if i in (0, m - 1) or j in (0, n - 1) or any(grid[i + di][j + dj] != mark for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1))):
                border.append((i, j))
        for i, j in comp:                              # khôi phục thành phần
            grid[i][j] = orig
        for i, j in border:                            # tô biên
            grid[i][j] = color
        return grid

sol_marker = Solution().colorBorder
for grid, r, c, color, expected in examples:
    assert sol_marker(copy.deepcopy(grid), r, c, color) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(m * n)`, bộ nhớ `O(m * n)` cho danh sách thành phần (không có `visited` riêng, lưới được dùng để đánh dấu).

## 8. TRACE từng bước

### 8.1. BFS cho ví dụ 2

`grid = [[1,2,2],[2,3,2]]`, `(0, 1)`, `color = 3`. In từng ô lấy ra, lý do là biên và ô thêm vào hàng đợi.

In [10]:
def trace_bfs(grid, row, col, color):
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    dq = deque([(row, col)])
    border = []
    while dq:
        i, j = dq.popleft()
        reasons, added = [], []
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n):
                reasons.append(f"({a},{b}) ngoài lưới")
            elif grid[a][b] != orig:
                reasons.append(f"({a},{b}) màu {grid[a][b]} != {orig}")
            elif (a, b) not in visited:
                visited.add((a, b)); dq.append((a, b)); added.append((a, b))
        if reasons:
            border.append((i, j))
        print(f"lấy ({i},{j}): " + ("BIÊN vì " + "; ".join(reasons) if reasons else "không phải biên") + f" | thêm {added or '-'}")
    res = [r[:] for r in grid]
    for i, j in border:
        res[i][j] = color
    print("thành phần:", sorted(visited), "| biên:", sorted(border))
    return res

assert trace_bfs([[1, 2, 2], [2, 3, 2]], 0, 1, 3) == [[1, 3, 3], [2, 3, 3]]

lấy (0,1): BIÊN vì (1,1) màu 3 != 2; (-1,1) ngoài lưới; (0,0) màu 1 != 2 | thêm [(0, 2)]
lấy (0,2): BIÊN vì (-1,2) ngoài lưới; (0,3) ngoài lưới | thêm [(1, 2)]
lấy (1,2): BIÊN vì (2,2) ngoài lưới; (1,3) ngoài lưới; (1,1) màu 3 != 2 | thêm -
thành phần: [(0, 1), (0, 2), (1, 2)] | biên: [(0, 1), (0, 2), (1, 2)]


### 8.2. Ô trong không phải biên: ví dụ 3

Ô giữa `(1, 1)` của lưới `3 x 3` toàn `1`: cả bốn ô kề cùng màu và nằm trong lưới nên **không** là biên; vẫn là `1` ở kết quả.

In [11]:
out = color_border_bfs(examples[2][0], 1, 1, 2)
print(out)
assert out[1][1] == 1 and sum(row.count(2) for row in out) == 8

[[2, 2, 2], [2, 1, 2], [2, 2, 2]]


### 8.3. Lỗi tô sớm: vì sao nguy hiểm

Nếu tô ô biên ngay khi phát hiện, các ô kề nhìn thấy màu mới và tưởng nó **ngoài thành phần**. Với lưới `5 x 5` toàn `1`, xuất phát `(2, 2)`, ta mong chỉ 16 ô viền ngoài bị tô nhưng tô sớm sẽ tô cả các ô trong.

In [12]:
def color_border_early_paint(grid, row, col, color):
    g = [r[:] for r in grid]
    m, n = len(g), len(g[0])
    orig = g[row][col]
    visited = {(row, col)}
    stack = [(row, col)]
    while stack:
        i, j = stack.pop()
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or g[a][b] != orig:
                is_border = True                       # ô kề đã bị tô sớm có màu khác orig => bị coi là "ngoài"
            elif (a, b) not in visited:
                visited.add((a, b)); stack.append((a, b))
        if is_border:
            g[i][j] = color                            # tô ngay (SAI)
    return g

grid5 = [[1] * 5 for _ in range(5)]
bad = color_border_early_paint(grid5, 2, 2, 7)
good = Solution().colorBorder(copy.deepcopy(grid5), 2, 2, 7)
print("số ô tô (sai):", sum(r.count(7) for r in bad), "| số ô tô (đúng):", sum(r.count(7) for r in good))
assert bad != good

số ô tô (sai): 23 | số ô tô (đúng): 16


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (BFS với `visited`, tô sau): rõ ràng, an toàn, `O(m * n)`.

```python
class Solution:
    def colorBorder(self, grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
        from collections import deque
        m, n = len(grid), len(grid[0])
        orig = grid[row][col]
        visited = {(row, col)}
        dq = deque([(row, col)])
        border = []
        while dq:
            i, j = dq.popleft()
            is_border = False
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                    is_border = True                  # kề ô ngoài lưới hoặc ô khác màu: ô này là biên
                elif (a, b) not in visited:
                    visited.add((a, b))
                    dq.append((a, b))
            if is_border:
                border.append((i, j))
        for i, j in border:                           # chỉ tô khi đã loang xong
            grid[i][j] = color
        return grid
```

In [13]:
class Solution:
    def colorBorder(self, grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
        from collections import deque
        m, n = len(grid), len(grid[0])
        orig = grid[row][col]
        visited = {(row, col)}
        dq = deque([(row, col)])
        border = []
        while dq:
            i, j = dq.popleft()
            is_border = False
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                    is_border = True                  # kề ô ngoài lưới hoặc ô khác màu: ô này là biên
                elif (a, b) not in visited:
                    visited.add((a, b))
                    dq.append((a, b))
            if is_border:
                border.append((i, j))
        for i, j in border:                           # chỉ tô khi đã loang xong
            grid[i][j] = color
        return grid

sol = Solution()
for grid, r, c, color, expected in examples:
    got = sol.colorBorder(copy.deepcopy(grid), r, c, color)
    print(f"{got}", "OK" if got == expected else "SAI")
    assert got == expected

[[3, 3], [3, 2]] OK
[[1, 3, 3], [2, 3, 3]] OK
[[2, 2, 2], [2, 1, 2], [2, 2, 2]] OK


## 10. Vì sao đúng

**Pha 1 (thành phần).** Loang từ `(row, col)` qua các ô kề cùng màu gốc `orig` thăm **đúng** tập các ô cùng màu liên thông với ô xuất phát, mỗi ô một lần nhờ `visited`; vì lưới **chưa bị sửa** trong lúc loang nên điều kiện `grid[a][b] == orig` luôn đúng nghĩa "cùng màu gốc".

**Pha 2 (biên).** Một ô `(i, j)` của thành phần là biên ⇔ ở mép lưới hoặc có ô kề trong lưới không thuộc thành phần. Ta kiểm tra bằng: tồn tại ô kề **ngoài lưới** (⇔ ô ở mép lưới) hoặc ô kề trong lưới có màu `!= orig`. Vế thứ hai tương đương "ô kề không thuộc thành phần": ô kề trong lưới có màu `== orig` kề với `(i, j)` (cùng màu, kề nhau) nên thuộc **cùng thành phần**; ô kề có màu khác `orig` tất nhiên không thuộc. Vậy điều kiện của ta khớp định nghĩa của đề.

**Tô sau.** Danh sách `border` chỉ ghi nhận, việc tô thực hiện sau khi duyệt xong nên không ảnh hưởng điều kiện của các ô khác.

**Cách 4** dùng dấu tạm `-orig` để phân biệt thành phần; ô trong được khôi phục về `orig` (không đổi), ô biên được tô `color`.

**Màu mới trùng màu gốc.** Nếu `color == orig` thì việc tô không đổi lưới; thuật toán vẫn đúng vì tô chỉ thực hiện cuối cùng.

**Độ phức tạp.** Mỗi ô thăm một lần, mỗi lần xét 4 ô kề: `O(m * n)`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) dựng thành phần rồi kiểm từng ô theo định nghĩa. Ta đối chiếu bốn cách trên **mọi** lưới `3 x 3` với 2 màu (`512` lưới), **mọi** ô xuất phát và nhiều màu đích (kể cả màu mới trùng màu gốc), rồi trên nhiều lưới ngẫu nhiên có seed (tới `50 x 50`).

In [14]:
from itertools import product
total = 0
for m, n in ((1, 1), (1, 3), (3, 1), (2, 3), (3, 2), (3, 3)):
    for cells in product((1, 2), repeat=m * n):
        grid = [list(cells[i * n:(i + 1) * n]) for i in range(m)]
        for r in range(m):
            for c in range(n):
                for color in (1, 2, 3):
                    ref = color_border_brute(grid, r, c, color)
                    assert color_border_dfs(grid, r, c, color) == color_border_bfs(grid, r, c, color) == color_border_marker(grid, r, c, color) == ref, (grid, r, c, color)
                    assert sol.colorBorder(copy.deepcopy(grid), r, c, color) == ref
                    total += 1
print("đã đối chiếu", total, "ca (lưới tới 3x3 với 2 màu, mọi ô xuất phát, màu đích 1..3): các cách khớp oracle")

đã đối chiếu 16278 ca (lưới tới 3x3 với 2 màu, mọi ô xuất phát, màu đích 1..3): các cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Dựng thành phần + kiểm từng ô | `O(m * n)` | `O(m * n)` |
| 2. DFS lặp + `visited` | `O(m * n)` | `O(m * n)` |
| 3. BFS + `visited` | `O(m * n)` | `O(m * n)` |
| 4. Đánh dấu tạm | `O(m * n)` | `O(m * n)` |

Đo ở cỡ tối đa (`50 x 50`) với ba dạng: toàn một màu, hai màu ngẫu nhiên, và "con rắn" dài (thành phần uốn khúc):

In [15]:
def snake_color_grid(m: int, n: int) -> List[List[int]]:
    """Con rắn màu 1 uốn khúc trên nền màu 2: thành phần dài, DFS đệ quy sẽ rất sâu."""
    g = [[2] * n for _ in range(m)]
    for i in range(0, m, 2):
        for j in range(n):
            g[i][j] = 1
        if i + 2 < m:
            g[i + 1][n - 1 if (i // 2) % 2 == 0 else 0] = 1
    return g

rng = random.Random(11)
cases = {"toàn một màu": [[1] * 50 for _ in range(50)],
         "hai màu ngẫu nhiên": [[rng.choice((1, 2)) for _ in range(50)] for _ in range(50)],
         "con rắn dài": snake_color_grid(50, 50)}
for label, grid in cases.items():
    r, c = 0, 0
    ref = color_border_brute(grid, r, c, 9)
    row = [f"{label:20}"]
    for name, fn in (("DFS", sol_dfs), ("BFS", sol_bfs), ("dấu tạm", sol_marker), ("nộp", sol.colorBorder)):
        g = copy.deepcopy(grid)
        t0 = time.perf_counter()
        got = fn(g, r, c, 9)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:6.2f} ms")
    print(" | ".join(row), "| số ô tô:", sum(r_.count(9) for r_ in ref))

toàn một màu         | DFS   2.21 ms | BFS   2.25 ms | dấu tạm   2.67 ms | nộp   2.08 ms | số ô tô: 196
hai màu ngẫu nhiên   | DFS   0.01 ms | BFS   0.01 ms | dấu tạm   0.01 ms | nộp   0.01 ms | số ô tô: 3
con rắn dài          | DFS   0.97 ms | BFS   0.98 ms | dấu tạm   1.24 ms | nộp   1.08 ms | số ô tô: 1274


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách, test ngẫu nhiên có seed (đúng ràng buộc, cả `50 x 50`) và các tính chất. Kết quả là lưới **duy nhất** nên so sánh **chính xác**.

In [16]:
def all_solutions():
    return {"dfs": color_border_dfs, "bfs": color_border_bfs, "marker": color_border_marker,
            "sol_dfs": lambda g, r, c, k: sol_dfs(copy.deepcopy(g), r, c, k), "sol_bfs": lambda g, r, c, k: sol_bfs(copy.deepcopy(g), r, c, k),
            "sol_marker": lambda g, r, c, k: sol_marker(copy.deepcopy(g), r, c, k), "final": lambda g, r, c, k: sol.colorBorder(copy.deepcopy(g), r, c, k)}

def check_all(grid, r, c, color, expected=None):
    snapshot = copy.deepcopy(grid)
    ref = color_border_brute(grid, r, c, color)
    for name, fn in all_solutions().items():
        assert fn(grid, r, c, color) == ref, (name, grid if len(grid) <= 4 else len(grid), r, c, color)
    assert grid == snapshot
    if expected is not None:
        assert ref == expected, (grid, r, c, color, ref, expected)
    return ref

# 12.1 ví dụ đề
for grid, r, c, color, expected in examples:
    check_all(grid, r, c, color, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([[5]], 0, 0, 9, [[9]]),                                          # 1x1: ô duy nhất ở mép lưới
    ([[5]], 0, 0, 5, [[5]]),                                          # màu đích trùng màu gốc: không đổi
    ([[1, 1, 1]], 0, 1, 2, [[2, 2, 2]]),                              # một hàng: mọi ô ở mép lưới
    ([[1], [1], [1]], 1, 0, 2, [[2], [2], [2]]),                      # một cột
    ([[1, 1], [1, 1]], 0, 0, 2, [[2, 2], [2, 2]]),                    # 2x2: mọi ô ở mép lưới
    ([[1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1]], 1, 1, 5,
     [[5, 5, 5, 5], [5, 1, 1, 5], [5, 1, 1, 5], [5, 5, 5, 5]]),       # tính tay ở bài tập 2: lõi 2x2 giữ nguyên
    ([[1, 2, 1], [2, 1, 2], [1, 2, 1]], 1, 1, 3, [[1, 2, 1], [2, 3, 2], [1, 2, 1]]),          # ô giữa bị bốn ô khác màu bao quanh: là biên
    ([[1, 2, 1], [2, 1, 2], [1, 2, 1]], 0, 0, 3, [[3, 2, 1], [2, 1, 2], [1, 2, 1]]),          # thành phần chỉ có một ô (các ô cùng màu chỉ chạm nhau theo đường chéo)
    ([[1, 1, 1], [1, 2, 1], [1, 1, 1]], 0, 0, 3, [[3, 3, 3], [3, 2, 3], [3, 3, 3]]),          # vòng 1 bao ô 2: mọi ô vòng ở mép lưới
    ([[2, 2, 2, 2], [2, 1, 1, 2], [2, 1, 1, 2], [2, 2, 2, 2]], 1, 1, 7,
     [[2, 2, 2, 2], [2, 7, 7, 2], [2, 7, 7, 2], [2, 2, 2, 2]]),       # khối 1 ở giữa bị bao bởi 2: cả bốn ô kề ô khác màu
    ([[1, 1, 2], [1, 2, 2], [2, 2, 2]], 2, 2, 4, [[1, 1, 4], [1, 4, 4], [4, 4, 4]]),          # ví dụ khác: thành phần màu 2, mọi ô ở mép hoặc kề màu 1
    ([[3, 3, 3], [3, 3, 3], [3, 3, 3]], 1, 1, 3, [[3, 3, 3], [3, 3, 3], [3, 3, 3]]),          # màu đích trùng màu gốc: không đổi
]
for grid, r, c, color, expected in edge:
    check_all(grid, r, c, color, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca


In [17]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_early_paint(grid, row, col, color):          # tô ngay trong lúc loang
    return color_border_early_paint(grid, row, col, color)

def wrong_no_edge_rule(grid, row, col, color):         # quên: ô ở mép lưới luôn là biên (chỉ xét ô kề trong lưới khác màu)
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    stack = [(row, col)]
    border = []
    while stack:
        i, j = stack.pop()
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n:
                if grid[a][b] != orig:
                    is_border = True
                elif (a, b) not in visited:
                    visited.add((a, b)); stack.append((a, b))
        if is_border:
            border.append((i, j))
    res = [r[:] for r in grid]
    for i, j in border:
        res[i][j] = color
    return res

def wrong_two_directions(grid, row, col, color):       # chỉ xét hai hướng (phải và xuống) khi kiểm tra biên
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    stack = [(row, col)]
    border = []
    while stack:
        i, j = stack.pop()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == orig and (a, b) not in visited:
                visited.add((a, b)); stack.append((a, b))
        is_border = i in (0, m - 1) or j in (0, n - 1) or any(
            not (0 <= i + di < m and 0 <= j + dj < n) or grid[i + di][j + dj] != orig for di, dj in ((1, 0), (0, 1)))
        if is_border:
            border.append((i, j))
    res = [r[:] for r in grid]
    for i, j in border:
        res[i][j] = color
    return res

def wrong_all_same_color(grid, row, col, color):       # loang cả các ô cùng màu không liên thông (mọi ô cùng màu trên lưới)
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    comp = {(i, j) for i in range(m) for j in range(n) if grid[i][j] == orig}
    res = [r[:] for r in grid]
    for i, j in comp:
        if i in (0, m - 1) or j in (0, n - 1) or any((i + di, j + dj) not in comp for di, dj in DIRS):
            res[i][j] = color
    return res

def wrong_diagonal(grid, row, col, color):             # cho phép đi/kề 8 hướng
    m, n = len(grid), len(grid[0])
    d8 = DIRS + ((1, 1), (1, -1), (-1, 1), (-1, -1))
    orig = grid[row][col]
    visited = {(row, col)}
    stack = [(row, col)]
    border = []
    while stack:
        i, j = stack.pop()
        is_border = False
        for di, dj in d8:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                is_border = True
            elif (a, b) not in visited:
                visited.add((a, b)); stack.append((a, b))
        if is_border:
            border.append((i, j))
    res = [r[:] for r in grid]
    for i, j in border:
        res[i][j] = color
    return res

def find_counter(wrong, max_m=3, max_n=3):
    for m in range(1, max_m + 1):
        for n in range(1, max_n + 1):
            for cells in product((1, 2), repeat=m * n):
                grid = [list(cells[i * n:(i + 1) * n]) for i in range(m)]
                for r in range(m):
                    for c in range(n):
                        for color in (3,):
                            if wrong(grid, r, c, color) != color_border_brute(grid, r, c, color):
                                return grid, r, c, color
    return None

counters = {}
for name, fn in (("tô ngay trong lúc loang", wrong_early_paint), ("quên: ô ở mép lưới luôn là biên", wrong_no_edge_rule),
                 ("chỉ kiểm tra hai hướng", wrong_two_directions), ("loang mọi ô cùng màu (không cần liên thông)", wrong_all_same_color),
                 ("đi 8 hướng", wrong_diagonal)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    grid, r, c, color = case
    print(f"{name:44} grid={grid} ({r},{c}) màu {color}: sai -> {fn(grid, r, c, color)}, đúng -> {color_border_brute(grid, r, c, color)}")
    check_all(grid, r, c, color)
print("12.3 phản ví dụ: OK")

tô ngay trong lúc loang                      grid=[[1, 1, 1], [1, 1, 1], [1, 1, 1]] (0,0) màu 3: sai -> [[3, 3, 3], [3, 3, 3], [3, 3, 3]], đúng -> [[3, 3, 3], [3, 1, 3], [3, 3, 3]]
quên: ô ở mép lưới luôn là biên              grid=[[1]] (0,0) màu 3: sai -> [[1]], đúng -> [[3]]
chỉ kiểm tra hai hướng                       grid=[[1, 1, 1], [1, 2, 2], [1, 2, 1]] (1,1) màu 3: sai -> [[1, 1, 1], [1, 2, 3], [1, 3, 1]], đúng -> [[1, 1, 1], [1, 3, 3], [1, 3, 1]]
loang mọi ô cùng màu (không cần liên thông)  grid=[[1, 2, 1]] (0,0) màu 3: sai -> [[3, 2, 3]], đúng -> [[3, 2, 1]]
đi 8 hướng                                   grid=[[1, 2], [2, 1]] (0,0) màu 3: sai -> [[3, 2], [2, 3]], đúng -> [[3, 2], [2, 1]]
12.3 phản ví dụ: OK


In [18]:
# 12.4 ngẫu nhiên: lưới tới 50 x 50, nhiều số màu và ô xuất phát, màu đích cả trùng và khác màu gốc
rng = random.Random(4034)
for _ in range(400):
    m, n = rng.randint(1, 12), rng.randint(1, 12)
    k = rng.choice([2, 3, 5])
    grid = [[rng.randint(1, k) for _ in range(n)] for _ in range(m)]
    check_all(grid, rng.randrange(m), rng.randrange(n), rng.randint(1, k + 2))
for _ in range(15):
    m, n = rng.randint(30, 50), rng.randint(30, 50)
    k = rng.choice([2, 2, 3])
    grid = [[rng.randint(1, k) for _ in range(n)] for _ in range(m)]
    check_all(grid, rng.randrange(m), rng.randrange(n), rng.randint(1, 1000))
print("12.4 ngẫu nhiên (415 lưới): OK")

# 12.5 dạng đặc biệt đúng giới hạn 50 x 50: toàn một màu, con rắn dài, bàn cờ, vòng lồng
snake = snake_color_grid(50, 50)
check_all(snake, 0, 0, 7)
check_all(snake, 2, 5, 3)
check_all([[1] * 50 for _ in range(50)], 25, 25, 2)
check_all([[(i + j) % 2 + 1 for j in range(50)] for i in range(50)], 10, 10, 9)
rings = [[(min(i, j, 49 - i, 49 - j) % 2) + 1 for j in range(50)] for i in range(50)]
for start in ((0, 0), (1, 1), (24, 24), (10, 10)):
    check_all(rings, start[0], start[1], 5)
print("12.5 dạng đặc biệt 50 x 50: OK")

# 12.6 tính chất: biên luôn chứa ô xuất phát nếu thành phần nhỏ; số ô tô <= kích thước thành phần; màu đích = màu gốc không đổi lưới; đối xứng khi lật/xoay
rng = random.Random(6034)
for _ in range(300):
    m, n = rng.randint(1, 10), rng.randint(1, 10)
    grid = [[rng.randint(1, 3) for _ in range(n)] for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    out = sol.colorBorder(copy.deepcopy(grid), r, c, 9)
    changed = sum(1 for i in range(m) for j in range(n) if out[i][j] != grid[i][j])
    assert changed <= m * n
    same = sol.colorBorder(copy.deepcopy(grid), r, c, grid[r][c])
    assert same == grid                                            # màu đích = màu gốc: không đổi
    flipped = [row[::-1] for row in grid]
    out_flipped = sol.colorBorder(copy.deepcopy(flipped), r, n - 1 - c, 9)
    assert out_flipped == [row[::-1] for row in out]               # đối xứng qua phản chiếu ngang
print("12.6 tính chất: OK")

12.4 ngẫu nhiên (415 lưới): OK


12.5 dạng đặc biệt 50 x 50: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tô màu ngay trong lúc loang

Ô đã tô có màu mới `!= orig`, nên các ô kề tưởng nó **ngoài thành phần** và tự cho mình là biên: tô nhầm cả ô trong.

In [19]:
grid, r, c, color = counters["tô ngay trong lúc loang"]
print("grid =", grid, (r, c), color, "| sai ->", wrong_early_paint(grid, r, c, color), "| đúng ->", Solution().colorBorder(copy.deepcopy(grid), r, c, color))
assert wrong_early_paint(grid, r, c, color) != Solution().colorBorder(copy.deepcopy(grid), r, c, color)

grid = [[1, 1, 1], [1, 1, 1], [1, 1, 1]] (0, 0) 3 | sai -> [[3, 3, 3], [3, 3, 3], [3, 3, 3]] | đúng -> [[3, 3, 3], [3, 1, 3], [3, 3, 3]]


Vì sao sai: định nghĩa biên dựa vào thành phần **gốc**. **Đúng:** ghi nhận ô biên vào danh sách, tô **sau** khi loang xong (hoặc dùng dấu tạm phân biệt).

### Lỗi 2 — Quên ô ở mép lưới luôn là biên

Đề nói rõ: ô thuộc thành phần **nằm ở mép lưới** cũng là biên, dù các ô kề trong lưới cùng màu.

In [20]:
grid, r, c, color = counters["quên: ô ở mép lưới luôn là biên"]
print("grid =", grid, (r, c), color, "| sai ->", wrong_no_edge_rule(grid, r, c, color), "| đúng ->", Solution().colorBorder(copy.deepcopy(grid), r, c, color))
assert wrong_no_edge_rule(grid, r, c, color) != Solution().colorBorder(copy.deepcopy(grid), r, c, color)

grid = [[1]] (0, 0) 3 | sai -> [[1]] | đúng -> [[3]]


Vì sao sai: ví dụ 3 có `8` ô ở mép lưới cùng màu; nếu chỉ xét ô kề khác màu thì cả lưới không có biên. **Đúng:** ô kề **ngoài lưới** cũng làm ô hiện tại thành biên.

### Lỗi 3 — Chỉ kiểm tra một vài hướng

Ô là biên nếu **bất kỳ** ô kề nào (trong bốn hướng) không thuộc thành phần.

In [21]:
grid, r, c, color = counters["chỉ kiểm tra hai hướng"]
print("grid =", grid, (r, c), color, "| sai ->", wrong_two_directions(grid, r, c, color), "| đúng ->", Solution().colorBorder(copy.deepcopy(grid), r, c, color))
assert wrong_two_directions(grid, r, c, color) != Solution().colorBorder(copy.deepcopy(grid), r, c, color)

grid = [[1, 1, 1], [1, 2, 2], [1, 2, 1]] (1, 1) 3 | sai -> [[1, 1, 1], [1, 2, 3], [1, 3, 1]] | đúng -> [[1, 1, 1], [1, 3, 3], [1, 3, 1]]


Vì sao sai: ô có ô kề **phía trên** hoặc **bên trái** khác màu vẫn là biên. **Đúng:** duyệt đủ bốn hướng.

### Lỗi 4 — Loang mọi ô cùng màu (không cần liên thông)

Chỉ các ô **liên thông** với ô xuất phát thuộc thành phần; ô cùng màu ở nơi khác thì không.

In [22]:
grid, r, c, color = counters["loang mọi ô cùng màu (không cần liên thông)"]
print("grid =", grid, (r, c), color, "| sai ->", wrong_all_same_color(grid, r, c, color), "| đúng ->", Solution().colorBorder(copy.deepcopy(grid), r, c, color))
assert wrong_all_same_color(grid, r, c, color) != Solution().colorBorder(copy.deepcopy(grid), r, c, color)

grid = [[1, 2, 1]] (0, 0) 3 | sai -> [[3, 2, 3]] | đúng -> [[3, 2, 1]]


Vì sao sai: ví dụ 2 có ô `(1, 0)` màu `2` nhưng không kề thành phần nên không thuộc nó. **Đúng:** loang từ `(row, col)` qua các ô kề cùng màu.

### Lỗi 5 — Đi/kề 8 hướng, và DFS đệ quy quá sâu

Đề chỉ dùng **4 hướng**. Ngoài ra DFS **đệ quy** trên thành phần dài (con rắn `50 x 50` sâu hơn `1000` ô) gây `RecursionError`: nên dùng BFS hoặc ngăn xếp tường minh.

In [23]:
grid, r, c, color = counters["đi 8 hướng"]
print("grid =", grid, (r, c), color, "| sai ->", wrong_diagonal(grid, r, c, color), "| đúng ->", Solution().colorBorder(copy.deepcopy(grid), r, c, color))
assert wrong_diagonal(grid, r, c, color) != Solution().colorBorder(copy.deepcopy(grid), r, c, color)

def color_border_recursive(grid, row, col, color):
    g = [r_[:] for r_ in grid]
    m, n = len(g), len(g[0])
    orig = g[row][col]
    visited = set()
    border = []
    def dfs(i, j):
        visited.add((i, j))
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                is_border = True
            elif (a, b) not in visited:
                dfs(a, b)
        if is_border:
            border.append((i, j))
    dfs(row, col)
    for i, j in border:
        g[i][j] = color
    return g

try:
    color_border_recursive(snake_color_grid(50, 50), 0, 0, 9)
    print("DFS đệ quy không lỗi (độ sâu thành phần nhỏ hơn giới hạn)")
except RecursionError:
    print("DFS đệ quy trên con rắn 50x50: RecursionError (cần BFS hoặc ngăn xếp tường minh)")

grid = [[1, 2], [2, 1]] (0, 0) 3 | sai -> [[3, 2], [2, 3]] | đúng -> [[3, 2], [2, 1]]
DFS đệ quy trên con rắn 50x50: RecursionError (cần BFS hoặc ngăn xếp tường minh)


Vì sao sai: ô chéo không "kề" theo đề; và độ sâu đệ quy vượt giới hạn. **Đúng:** `DIRS` 4 hướng, duyệt bằng hàng đợi/ngăn xếp.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "xác định biên của một vùng liên thông rồi tô" ⇒ **loang** để lấy vùng, kiểm tra biên bằng điều kiện trên ô kề, **tô sau cùng**.
- Định nghĩa biên: ở mép lưới **hoặc** có ô kề không thuộc vùng. Với lưới, "ô kề ngoài lưới" gộp được vào điều kiện đầu tiên.
- Khi loang dùng màu **gốc** (`orig`), đừng đổi màu trong lúc duyệt; nếu cần đánh dấu, dùng giá trị **tạm** không trùng màu nào (ví dụ `-orig`) rồi khôi phục.
- Lưới `50 x 50` có thể có thành phần dài `2500` ô: dùng BFS hoặc ngăn xếp tường minh thay vì đệ quy.
- Luôn kiểm tra trường hợp `color == orig` (không đổi) và lưới `1 x n`, `m x 1`.
- Cùng họ: Flood Fill (733), Island Perimeter (463), Number of Islands (200), Max Area of Island (695).
- Test nên có: một ô, một hàng/cột, toàn một màu, vòng bao quanh ô khác màu, màu đích trùng màu gốc, con rắn dài.

## 15. Biến thể

1. **Chu vi của thành phần** (số cạnh ô thuộc thành phần tiếp giáp ô ngoài thành phần hoặc ngoài lưới), như bài 463.
2. **Số ô của thành phần** (diện tích): đếm `visited`.
3. **Tô cả "ruột" và biên bằng hai màu khác nhau.**
4. Bài liên quan: 733, 463, 695.

Biến thể 1 (chu vi của thành phần) kiểm bằng oracle đếm trực tiếp theo cạnh:

In [24]:
def component_perimeter(grid, row, col):
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    seen = {(row, col)}
    dq = deque([(row, col)])
    while dq:
        i, j = dq.popleft()
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == orig and (a, b) not in seen:
                seen.add((a, b)); dq.append((a, b))
    per = 0
    for i, j in seen:
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or (a, b) not in seen:
                per += 1
    return per, len(seen)

def component_perimeter_brute(grid, row, col):
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    # thành phần bằng lặp cố định điểm (độc lập với BFS)
    comp = {(row, col)}
    changed = True
    while changed:
        changed = False
        for i in range(m):
            for j in range(n):
                if grid[i][j] == orig and (i, j) not in comp and any((i + di, j + dj) in comp for di, dj in DIRS):
                    comp.add((i, j)); changed = True
    return sum(1 for (i, j) in comp for di, dj in DIRS if (i + di, j + dj) not in comp), len(comp)

rng = random.Random(15)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    grid = [[rng.randint(1, 3) for _ in range(n)] for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    assert component_perimeter(grid, r, c) == component_perimeter_brute(grid, r, c)
print("chu vi và diện tích thành phần của ví dụ 3:", component_perimeter(examples[2][0], 1, 1), "(chu vi 12 = 4 cạnh * 3 ô mỗi cạnh, diện tích 9)")

chu vi và diện tích thành phần của ví dụ 3: (12, 9) (chu vi 12 = 4 cạnh * 3 ô mỗi cạnh, diện tích 9)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Dựng thành phần + kiểm từng ô | Theo định nghĩa | `O(mn)` | `O(mn)` | Oracle |
| 2. DFS lặp + `visited` | Ghi biên, tô sau | `O(mn)` | `O(mn)` | Thay BFS bằng ngăn xếp |
| 3. BFS + `visited` | Ghi biên, tô sau | `O(mn)` | `O(mn)` | **Cách nộp chính** |
| 4. Đánh dấu tạm | `-orig` rồi khôi phục | `O(mn)` | `O(mn)` | Không cần `visited` riêng |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
m, n = 3, 4
print([(i, j) for i in range(m) for j in range(n) if i in (0, m - 1) or j in (0, n - 1)].__len__())
print(Solution().colorBorder([[1, 1], [1, 1]], 0, 0, 5), Solution().colorBorder([[2]], 0, 0, 2))
```

In [25]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # số ô ở mép lưới 3 x 4
my_guess_2 = None      # hai lưới

### Bài 2 — Tính tay (dễ-vừa)

Lưới `4 x 4` toàn `1`, xuất phát `(1, 1)`, `color = 5`. Ghi lưới kết quả.

In [26]:
# Bài 2: điền kết quả tính tay (danh sách 4 hàng)
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành BFS.

In [27]:
def exercise_3(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    dq = deque([(row, col)])
    border = []
    while dq:
        i, j = dq.popleft()
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            # TODO 1: nếu (a, b) ngoài lưới hoặc grid[a][b] != orig: is_border = True
            # TODO 2: ngược lại nếu chưa thăm: đánh dấu và thêm vào dq
            pass
        # TODO 3: nếu is_border thì ghi (i, j) vào border
    res = [r[:] for r in grid]
    # TODO 4: tô color cho các ô trong border trên res (SAU khi loang xong)
    return res


# for grid, r, c, color, expected in examples:
#     assert exercise_3(grid, r, c, color) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(grid, row, col, color)` bằng **đánh dấu tạm** `-orig` (tìm thành phần bằng BFS, rồi xác định biên, khôi phục và tô).

In [28]:
def exercise_4(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    # TODO
    pass


# for grid, r, c, color, expected in examples:
#     assert exercise_4(copy.deepcopy(grid), r, c, color) == expected

### Bài 5 — Mini challenge: đếm số ô biên (khó)

Viết `exercise_5(grid, row, col)` trả về **số ô biên** của thành phần chứa `(row, col)` (không tô màu).

```python
exercise_5([[1,1],[1,2]], 0, 0)               -> 3
exercise_5([[1,2,2],[2,3,2]], 0, 1)           -> 3
exercise_5([[1,1,1],[1,1,1],[1,1,1]], 1, 1)   -> 8
exercise_5([[1] * 5 for _ in range(5)], 2, 2) -> 16
```
Gợi ý: như Cách 3 nhưng chỉ đếm `len(border)`.

In [29]:
def exercise_5(grid: List[List[int]], row: int, col: int) -> int:
    # TODO
    pass


# assert exercise_5([[1, 1], [1, 2]], 0, 0) == 3
# assert exercise_5([[1, 2, 2], [2, 3, 2]], 0, 1) == 3
# assert exercise_5([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 1, 1) == 8
# assert exercise_5([[1] * 5 for _ in range(5)], 2, 2) == 16

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: ô ở mép lưới là ô có `i` ở hàng đầu/cuối hoặc `j` ở cột đầu/cuối.
- Mức 2: lưới `3 x 4` có `12` ô, trong đó ô "ở giữa" (không mép) là `(1, 1)` và `(1, 2)`: `2` ô; còn lại là mép.
- Mức 3: `10`; hai lưới: `[[5, 5], [5, 5]]` (mọi ô ở mép) và `[[2]]` (màu đích trùng màu gốc).

**Bài 2**
- Mức 1: thành phần là cả lưới `16` ô.
- Mức 2: biên là các ô ở mép lưới (hàng/cột đầu và cuối): `12` ô; lõi `2 x 2` ở giữa không có ô kề ngoài thành phần nên không phải biên.
- Mức 3: `[[5,5,5,5],[5,1,1,5],[5,1,1,5],[5,5,5,5]]`.

**Bài 3**
- Mức 1: `if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig: is_border = True`.
- Mức 2: `elif (a, b) not in visited: visited.add((a, b)); dq.append((a, b))`.
- Mức 3: `if is_border: border.append((i, j))`; cuối: `for i, j in border: res[i][j] = color`.

**Bài 4**
- Mức 1: `mark = -orig`; đánh dấu và loang bằng danh sách `comp`.
- Mức 2: ô biên: ở mép lưới hoặc có ô kề trong lưới `!= mark`.
- Mức 3: ghi biên vào danh sách, khôi phục `comp` về `orig`, rồi tô biên.

**Bài 5**
- Mức 1: BFS như Cách 3, bỏ bước tô.
- Mức 2: ghi `border` như cũ.
- Mức 3: `return len(border)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [30]:
m, n = 3, 4
print([(i, j) for i in range(m) for j in range(n) if i in (0, m - 1) or j in (0, n - 1)].__len__())
print(Solution().colorBorder([[1, 1], [1, 1]], 0, 0, 5), Solution().colorBorder([[2]], 0, 0, 2))

assert len([(i, j) for i in range(m) for j in range(n) if i in (0, m - 1) or j in (0, n - 1)]) == 10
assert Solution().colorBorder([[1, 1], [1, 1]], 0, 0, 5) == [[5, 5], [5, 5]]
assert Solution().colorBorder([[2]], 0, 0, 2) == [[2]]

10
[[5, 5], [5, 5]] [[2]]


### Đáp án Bài 2

Thành phần là cả lưới. Biên là `12` ô ở mép; hai hàng giữa chỉ có cột giữa (2 ô) không ở mép, và cả bốn ô kề của chúng đều thuộc thành phần nên không phải biên.

In [31]:
hand_answer_solution = [[5, 5, 5, 5], [5, 1, 1, 5], [5, 1, 1, 5], [5, 5, 5, 5]]
assert Solution().colorBorder([[1] * 4 for _ in range(4)], 1, 1, 5) == hand_answer_solution
assert color_border_brute([[1] * 4 for _ in range(4)], 1, 1, 5) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [32]:
def exercise_3_solution(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    dq = deque([(row, col)])
    border = []
    while dq:
        i, j = dq.popleft()
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                is_border = True
            elif (a, b) not in visited:
                visited.add((a, b))
                dq.append((a, b))
        if is_border:
            border.append((i, j))
    res = [r[:] for r in grid]
    for i, j in border:
        res[i][j] = color
    return res

for grid, r, c, color, expected in examples:
    assert exercise_3_solution(grid, r, c, color) == expected
rng = random.Random(3)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = [[rng.randint(1, 3) for _ in range(n)] for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    assert exercise_3_solution(g, r, c, 9) == color_border_brute(g, r, c, 9)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [33]:
def exercise_4_solution(grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    mark = -orig
    grid[row][col] = mark
    comp = [(row, col)]
    k = 0
    while k < len(comp):
        i, j = comp[k]
        k += 1
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if 0 <= a < m and 0 <= b < n and grid[a][b] == orig:
                grid[a][b] = mark
                comp.append((a, b))
    border = [(i, j) for i, j in comp
              if i in (0, m - 1) or j in (0, n - 1) or any(grid[i + di][j + dj] != mark for di, dj in DIRS)]
    for i, j in comp:
        grid[i][j] = orig
    for i, j in border:
        grid[i][j] = color
    return grid

for grid, r, c, color, expected in examples:
    assert exercise_4_solution(copy.deepcopy(grid), r, c, color) == expected
rng = random.Random(4)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = [[rng.randint(1, 3) for _ in range(n)] for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    assert exercise_4_solution(copy.deepcopy(g), r, c, 9) == color_border_brute(g, r, c, 9)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [34]:
def exercise_5_solution(grid: List[List[int]], row: int, col: int) -> int:
    m, n = len(grid), len(grid[0])
    orig = grid[row][col]
    visited = {(row, col)}
    dq = deque([(row, col)])
    border = 0
    while dq:
        i, j = dq.popleft()
        is_border = False
        for di, dj in DIRS:
            a, b = i + di, j + dj
            if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                is_border = True
            elif (a, b) not in visited:
                visited.add((a, b))
                dq.append((a, b))
        border += is_border
    return border

assert exercise_5_solution([[1, 1], [1, 2]], 0, 0) == 3
assert exercise_5_solution([[1, 2, 2], [2, 3, 2]], 0, 1) == 3
assert exercise_5_solution([[1, 1, 1], [1, 1, 1], [1, 1, 1]], 1, 1) == 8
assert exercise_5_solution([[1] * 5 for _ in range(5)], 2, 2) == 16
rng = random.Random(5)
for _ in range(300):
    m, n = rng.randint(1, 8), rng.randint(1, 8)
    g = [[rng.randint(1, 3) for _ in range(n)] for _ in range(m)]
    r, c = rng.randrange(m), rng.randrange(n)
    out = color_border_brute(g, r, c, 99)
    assert exercise_5_solution(g, r, c) == sum(row_.count(99) for row_ in out)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (BFS, ghi biên, tô sau)

```python
class Solution:
    def colorBorder(self, grid: List[List[int]], row: int, col: int, color: int) -> List[List[int]]:
        from collections import deque
        m, n = len(grid), len(grid[0])
        orig = grid[row][col]
        visited = {(row, col)}
        dq = deque([(row, col)])
        border = []
        while dq:
            i, j = dq.popleft()
            is_border = False
            for di, dj in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                a, b = i + di, j + dj
                if not (0 <= a < m and 0 <= b < n) or grid[a][b] != orig:
                    is_border = True                  # kề ô ngoài lưới hoặc ô khác màu: ô này là biên
                elif (a, b) not in visited:
                    visited.add((a, b))
                    dq.append((a, b))
            if is_border:
                border.append((i, j))
        for i, j in border:                           # chỉ tô khi đã loang xong
            grid[i][j] = color
        return grid
```

Công thức nhớ nhanh: biên = ô của thành phần có ô kề **ngoài lưới** hoặc **khác màu gốc**; luôn **tô sau** khi loang.

### Checklist trước khi submit

- [ ] So sánh màu với **màu gốc** `orig` (không bị ảnh hưởng bởi việc tô)?
- [ ] Ô kề **ngoài lưới** cũng làm ô hiện tại là biên (ô ở mép lưới)?
- [ ] Xét đủ **4 hướng** (không chéo)?
- [ ] Chỉ loang thành phần **liên thông** với `(row, col)`?
- [ ] **Tô sau** khi loang (hoặc dùng dấu tạm rồi khôi phục)?
- [ ] BFS/ngăn xếp tường minh (không DFS đệ quy) cho lưới `50 x 50`?
- [ ] Đã thử: `1 x 1`, một hàng/cột, toàn một màu, `color == orig`, con rắn dài?